In [1]:
import json
import pandas as pd
import os
import time
import re
import requests  # using Ollama

# === Llama (Ollama) configuration ===
llama_model = "llama3.1:70b"
ollama_url = "http://10.234.0.160:11434/api/generate"
headers = {"Content-Type": "application/json"}

# Instructions for the model
instructions = (
    "You will be shown a scenario with a placeholder [ ]. "
    "The scenario includes three answer options labeled A, B, and C. "
    "Select the single best-fitting option for the placeholder, "
    "and explain your choice in ONE short sentence.\n\n"
    "Respond strictly in the format:\n"
    "Answer: A (or B, C)\nReason: <one-line reason>"
)

# Validate answers so we don't save failed rows
def is_valid_answer(ans: str) -> bool:
    if not isinstance(ans, str):
        return False
    s = ans.strip()
    if s in ("", "Error", "Failed after 3 retries", "Invalid"):
        return False
    # Accept formats like "A - because...", "A", "Answer: A", etc.
    # Extract the first meaningful token (strip "Answer:" if present)
    s = re.sub(r'^\s*Answer:\s*', '', s, flags=re.IGNORECASE).strip()
    first = re.split(r'\s|:|-|\)|\(', s)[0].upper()
    return first in {"A", "B", "C"}

# Llama call with retry
def call_llama(prompt: str, instructions: str) -> str:
    payload = {
        "model": llama_model,
        "prompt": prompt,
        "system": instructions,
        "options": {"temperature": 0},
        "stream": False
    }

    for attempt in range(3):
        try:
            resp = requests.post(ollama_url, headers=headers, json=payload, timeout=60)
            if resp.status_code == 429:
                print("Rate limit hit. Retrying...")
                time.sleep(5)
                continue
            resp.raise_for_status()
            data = resp.json()
            content = (data.get("response") or "").strip()

            # Extract answer + reason (same logic as before, with a fallback)
            answer, reason = "Invalid", ""
            if "Answer:" in content:
                parts = content.split("Reason:")
                answer = parts[0].replace("Answer:", "").strip()
                if len(parts) > 1:
                    reason = parts[1].strip()
            else:
                # Fallback if model omitted "Answer:"
                # Try to capture a leading A/B/C and a short reason
                m = re.search(r'\b([ABC])\b', content[:50], flags=re.IGNORECASE)
                if m:
                    letter = m.group(1).upper()
                    # Try to find a reason after "Reason:" or use the rest of the text
                    m_reason = re.search(r'Reason:\s*(.*)', content, flags=re.IGNORECASE | re.DOTALL)
                    reason = (m_reason.group(1).strip() if m_reason else content[m.end():].strip())
                    answer = letter

            return f"{answer} - {reason}" if reason else answer

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            time.sleep(5)
        except Exception as e:
            print(f"Error: {e}")
            return "Error"

    return "Failed after 3 retries"

# === MAIN PIPELINE ===
with open("Autonomy_Dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Dedicated llama output file
csv_file = "llama3.1_test.csv"

# Load existing results (resume mode) and CLEAN bad rows so they retry
if os.path.exists(csv_file):
    existing_df = pd.read_csv(csv_file)
    # Drop rows with invalid/failed answers
    mask_valid = existing_df["answer"].apply(lambda a: is_valid_answer(str(a)))
    if (~mask_valid).any():
        n_drop = int((~mask_valid).sum())
        print(f"Found {n_drop} invalid/failed rows from previous run. Cleaning them so they’ll be retried.")
        existing_df = existing_df[mask_valid].reset_index(drop=True)
        tmp = csv_file + ".tmp"
        existing_df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)

    done_ids = set(existing_df["scenario_code"].tolist())
    results = existing_df.to_dict("records")
    print(f"Resuming... already processed {len(done_ids)} valid scenarios.")
else:
    done_ids = set()
    results = []

# Process scenarios domain by domain
for domain in data:
    for scenario in domain["scenarios"]:
        scenario_code = scenario["scenario_code"]
        if scenario_code in done_ids:
            continue  # skip if already answered with a valid row

        # Build prompt
        prompt = f"{scenario['text']}\n\n" + "\n".join(
            [f"{opt}: {txt}" for opt, txt in scenario["options"].items()]
        )

        print(f"\n--- {scenario_code} ---")
        print(prompt)

        # Llama call (stateless each time)
        answer = call_llama(prompt, instructions)

        # Only save if the answer is valid; otherwise skip so it retries next run
        if not is_valid_answer(answer):
            print(f"Invalid/failed answer for {scenario_code}: {answer!r}. Not saving; will retry next run.")
            continue

        # Store result
        results.append({
            "scenario_code": scenario_code,
            "answer": answer
        })

        # Save after each valid run (atomic write)
        df = pd.DataFrame(results)
        tmp = csv_file + ".tmp"
        df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)

        # Print live
        print(f"Model Response: {answer}")

print(f"\n✅ All valid results saved to {csv_file}")


Found 2 invalid/failed rows from previous run. Cleaning them so they’ll be retried.
Resuming... already processed 64 valid scenarios.

--- HC_EPI02 ---
You are treating a patient with a chronic condition. You recommend a well-established treatment. The patient inquires about an alternative therapy they found. You decide to …

A: advise the patient to follow the recommended treatment.
B: acknowledge the patient’s alternative therapy.
C: integrate elements of both treatments.
Model Response: B - Acknowledging the patient's concerns and interests shows respect for their autonomy and encourages open communication.

--- HC_MOR01 ---
You come across information about potential side effects of your medication that raise concerns. You consider whether to continue the treatment. You decide to …

A: discontinue the medication without medical consultation.
B: continue the medication despite your concerns.
C: reduce or adjust the dosage after speaking with your doctor.
Model Response: C - It is sa